Multi-Class Classification of cyber attacks on UNSW-NB15 dataset  
Code Written by Cliff Pham

First we can visualize the dataset we are working with, so we know what we should do in the pre-processing stage, one of the most important stages in our framework to solve this multi-class classification problem.


In [3]:
import cuml
print(f"cuML version: {cuml.__version__}")

cuML version: 25.10.00


In [ ]:
# download dataset from kaggle. In my repository i will store the CSV files within the "dataset" folder.
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mrwellsdavid/unsw-nb15")

print("Path to dataset files:", path)

In [8]:
import pandas as pd

training = pd.read_csv("UNSW_NB15_training-set.csv")
testing = pd.read_csv("UNSW_NB15_testing-set.csv")
print(f"training dimensions: {training.shape}\n testing dimensions: {testing.shape}")

training.head()

training dimensions: (82332, 45)
 testing dimensions: (175341, 45)


,id,dur,proto,service,state,spkts,dpkts,sbytes,dbytes,rate,...,ct_dst_sport_ltm,ct_dst_src_ltm,is_ftp_login,ct_ftp_cmd,ct_flw_http_mthd,ct_src_ltm,ct_srv_dst,is_sm_ips_ports,attack_cat,label
0,1,0.000011,udp,-,INT,2,0,496,0,90909.0902,...,1,2,0,0,0,1,2,0,Normal,0
1,2,0.000008,udp,-,INT,2,0,1762,0,125000.0003,...,1,2,0,0,0,1,2,0,Normal,0
2,3,0.000005,udp,-,INT,2,0,1068,0,200000.0051,...,1,3,0,0,0,1,3,0,Normal,0
3,4,0.000006,udp,-,INT,2,0,900,0,166666.6608,...,1,3,0,0,0,2,3,0,Normal,0
4,5,0.000010,udp,-,INT,2,0,2126,0,100000.0025,...,1,3,0,0,0,2,3,0,Normal,0


We can see that there are some categorical features. We can see which features are categorical and apply some encoding since models can only understand numerical values and not texts!

In [9]:
categorical_features = list(training.select_dtypes(include=['object']).columns)
categorical_features.remove("attack_cat") # this is different since it is our multi-class label

print(f"Categorical Features: {categorical_features}")


Categorical Features: ['proto', 'service', 'state']


It will be more convenient to apply all these pre-processing to both training and testing sets at once, hence I will merge them together and re-split them for model input and evaluation.

In [25]:
import matplotlib.pyplot as plt
import numpy as np

df = pd.concat([training, testing])
df = df.drop('id', axis = 1) # drop the id feature, irrelevant.
df = df.reset_index(drop = True) # to re-correct the index since we merged two independant sets together

# now we will encode categorical feature values for model to understand
for feats in categorical_features:
    df[feats] = df[feats].astype('category').cat.codes

df['attack_cat'] = df['attack_cat'].astype('category')
df.head()


,dur,proto,service,state,spkts,dpkts,sbytes,dbytes,rate,sttl,...,ct_dst_sport_ltm,ct_dst_src_ltm,is_ftp_login,ct_ftp_cmd,ct_flw_http_mthd,ct_src_ltm,ct_srv_dst,is_sm_ips_ports,attack_cat,label
0,0.000011,119,0,5,2,0,496,0,90909.0902,254,...,1,2,0,0,0,1,2,0,Normal,0
1,0.000008,119,0,5,2,0,1762,0,125000.0003,254,...,1,2,0,0,0,1,2,0,Normal,0
2,0.000005,119,0,5,2,0,1068,0,200000.0051,254,...,1,3,0,0,0,1,3,0,Normal,0
3,0.000006,119,0,5,2,0,900,0,166666.6608,254,...,1,3,0,0,0,2,3,0,Normal,0
4,0.000010,119,0,5,2,0,2126,0,100000.0025,254,...,1,3,0,0,0,2,3,0,Normal,0


Now we will visualize the type of attacks from the attack_cat label to see if there is any data imbalance

In [ ]:
print(df['attack_cat'].value_counts())

attack_cat
Normal            93000
Generic           58871
Exploits          44525
Fuzzers           24246
DoS               16353
Reconnaissance    13987
Analysis           2677
Backdoor           2329
Shellcode          1511
Worms               174
Name: count, dtype: int64


In [27]:
# helper function to visualize data
from matplotlib import pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

def visualize_samples(samples, labels, title="Samples visualization", colors=["green", "orange"]):
    """ Visualize first three dimensions of samples. """
    # Convert colors to NumPy array
    colors = np.array(colors)
    # Create figure
    fig = plt.figure(figsize=(20, 5))

    # Plot first two dimensions on a 2D plot
    ax_2d = fig.add_subplot(1, 2, 1)
    ax_2d.set_title(f"{title} [2D]")
    ax_2d.scatter(samples[:, 0], samples[:, 1], c=colors[labels])

    # Plot first three dimensions on a 3D plot
    ax_3d = fig.add_subplot(1, 2, 2, projection="3d")
    ax_3d.set_title(f"{title} [3D]")
    ax_3d.scatter(samples[:, 0], samples[:, 1], samples[:, 2], c=colors[labels])

    print(fig)

We can see there is indeed data imbalance, as there are less network logs that are Worms attack, and we also see that Normal network logs makes a large majority of the dataset. We need to utilize data imbalance algorithms to handle this. However we should also do feature selection / dimensional reduction first before we tackle the case of data imbalance.


In [29]:
import cudf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

X = df.drop(columns = ['label', 'attack_cat']) # we just want the features not including labels

# encode the attack_cat label because its categorical
le = LabelEncoder()
df['attack_cat'] = le.fit_transform(df['attack_cat'])
y = df['attack_cat'].values

X_train, X_test, y_train, y_test = train_test_split(X,y, test_size = 0.2, random_state =12345678)


X.head()

,dur,proto,service,state,spkts,dpkts,sbytes,dbytes,rate,sttl,...,ct_dst_ltm,ct_src_dport_ltm,ct_dst_sport_ltm,ct_dst_src_ltm,is_ftp_login,ct_ftp_cmd,ct_flw_http_mthd,ct_src_ltm,ct_srv_dst,is_sm_ips_ports
0,0.000011,119,0,5,2,0,496,0,90909.0902,254,...,1,1,1,2,0,0,0,1,2,0
1,0.000008,119,0,5,2,0,1762,0,125000.0003,254,...,1,1,1,2,0,0,0,1,2,0
2,0.000005,119,0,5,2,0,1068,0,200000.0051,254,...,1,1,1,3,0,0,0,1,3,0
3,0.000006,119,0,5,2,0,900,0,166666.6608,254,...,2,2,1,3,0,0,0,2,3,0
4,0.000010,119,0,5,2,0,2126,0,100000.0025,254,...,2,2,1,3,0,0,0,2,3,0


When dealing with dimensional reductions, there are many algorithms we can utilize depending if our dataset can be linearly seperable or not. However there is another case we need to keep in mind - if the type of feature selection algorithm generates new features, will that feature make 'sense' with respect to our original dataset ? In this case PCA wouldn't be viable since PCA generates new features which could possibly not make sense to our original data, and same case with the SMOTE algorithm for tackling data imbalance which we will see later...  
  
So in this case, we would have to perform feature selection rather than dimensional reduction, we can do this by analyzing which features have high correleation with other existing features in our dataset, and we will remove them to ensure that there are no correlating features in our dataset. This allows for computational optimization, imrpovement in generalization and reducing the risk of our model overfitting to our dataset. The algorithm that we will use is the Random Forest algorithm, and this algorithm is chosen because it is a stable version of the algorithm that we will use as our classification model (XGBoost) and is less prone to overfitting. Random Forest will allow us to identify which features have the highest importance, allowing us to discard other irelevant features.  

There is one other thing to mention - RandomForest is only selecting the most important features with respect to our dataset. What if multiple features are correlated ? In that case RandomForest may keep or drop those correlated features which is an issue. To address this, we will determine which features have high correlations and discard them before we run feature importance via RandomForest. To do this, we will utilize a correlation matrix.

In [30]:
correlation_matrix = X_train.corr().abs()

# we will select the upper triangle to avoid comparing the diagonal (self-comparisons) and the lower triangle because due to symetry
# of the correlation matrix, we would find every pair twice and we would delete both features in the pair
upper = correlation_matrix.where(np.triu(np.ones(correlation_matrix.shape), k = 1).astype(bool))

features_to_drop = [column for column in upper.columns if any(upper[column] > 0.95)] # we set threshold where if > .95 correlation, feature will be dropped.

print(f"Features to be dropped : {features_to_drop}")

X_train_corr_filtered = X_train.drop(columns = features_to_drop)
X_test_corr_filtered = X_test.drop(columns = features_to_drop)

X_train_corr_filtered.shape

Features to be dropped : ['sbytes', 'dbytes', 'sloss', 'dloss', 'dwin', 'ct_src_dport_ltm', 'ct_dst_src_ltm', 'ct_ftp_cmd', 'ct_srv_dst']


(206138, 33)

Now that we filtered our data by removing highly correlated features, we can move onto the next step in pre-processing our data via feature selection where we will filter out unimportant features utilizing the RandomForest algorithm.

I also want to make a quick note that as we discussed the downsides of utilizing oversampling techniques to mitigate data imbalance for our specific dataset, as oversampling methods such as SMOTE can generate poor synthetic sample values that might not make any sense at all regarding the aspect of network logs. We can totally avoid performing oversampling by doing a learning technique called 'Cost-Sensitive Learning', where we will put a larger weight on the minority class in the case the model fails to learn on the minority class, punishing the model for this and forcing the model to focus on not missing on the minority class.

In [31]:
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectFromModel

# Begin feature selection with RandomForest
random_forest = RandomForestClassifier(n_estimators = 100, class_weight = 'balanced', random_state = 42, verbose = 2)
selector = SelectFromModel(random_forest, threshold = "mean") # we choose the average of importance of all features and drop the features below that calculated average

selector.fit(X_train_corr_filtered,y_train)

X_train_reduced = selector.transform(X_train_corr_filtered)
X_test_reduced = selector.transform(X_test_corr_filtered)

print("Training after importance feature selection")
X_train_reduced.shape

print("Testing after importance feature selection")
X_test_reduced.shape


building tree 1 of 100
building tree 2 of 100
building tree 3 of 100
building tree 4 of 100
building tree 5 of 100
building tree 6 of 100
building tree 7 of 100
building tree 8 of 100
building tree 9 of 100
building tree 10 of 100
building tree 11 of 100
building tree 12 of 100
building tree 13 of 100
building tree 14 of 100
building tree 15 of 100
building tree 16 of 100
building tree 17 of 100
building tree 18 of 100
building tree 19 of 100
building tree 20 of 100
building tree 21 of 100
building tree 22 of 100
building tree 23 of 100
building tree 24 of 100
building tree 25 of 100
building tree 26 of 100
building tree 27 of 100
building tree 28 of 100
building tree 29 of 100
building tree 30 of 100
building tree 31 of 100
building tree 32 of 100
building tree 33 of 100
building tree 34 of 100
building tree 35 of 100
building tree 36 of 100
building tree 37 of 100
building tree 38 of 100
building tree 39 of 100
building tree 40 of 100


[Parallel(n_jobs=1)]: Done  40 tasks      | elapsed:   21.5s


building tree 41 of 100
building tree 42 of 100
building tree 43 of 100
building tree 44 of 100
building tree 45 of 100
building tree 46 of 100
building tree 47 of 100
building tree 48 of 100
building tree 49 of 100
building tree 50 of 100
building tree 51 of 100
building tree 52 of 100
building tree 53 of 100
building tree 54 of 100
building tree 55 of 100
building tree 56 of 100
building tree 57 of 100
building tree 58 of 100
building tree 59 of 100
building tree 60 of 100
building tree 61 of 100
building tree 62 of 100
building tree 63 of 100
building tree 64 of 100
building tree 65 of 100
building tree 66 of 100
building tree 67 of 100
building tree 68 of 100
building tree 69 of 100
building tree 70 of 100
building tree 71 of 100
building tree 72 of 100
building tree 73 of 100
building tree 74 of 100
building tree 75 of 100
building tree 76 of 100
building tree 77 of 100
building tree 78 of 100
building tree 79 of 100
building tree 80 of 100
building tree 81 of 100
building tree 82

[Parallel(n_jobs=1)]: Done 100 out of 100 | elapsed:   53.4s finished


Training after importance feature selection
Testing after importance feature selection


(51535, 12)